In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%reload_ext autoreload

In [0]:
from pyspark.sql.functions import col
from utils.config import catalog_name, schema_name, bronze_table, silver_table, checkpoint_silver_table
from transformations.silver_transformations import merge_silver_table,add_job_type

In [0]:
bronze_stream = (
    spark.readStream
         .option("readChangeFeed", "true")
         .table(f"{catalog_name}.{schema_name}.{bronze_table}")
         .filter(col("_change_type").isin("insert", "update_postimage"))
)

# merge into silver table
(
    bronze_stream.writeStream
        .foreachBatch(merge_silver_table)
        .option("checkpointLocation", checkpoint_silver_table)
        .trigger(once=True)
        .start()
)
